In [1]:
# This notebook is used to find a network from a list of nodes in Translator

In [2]:
import sys
sys.path.append('../src')
import TCT as TCT
import pandas as pd

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT



In [3]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources(use_new_metakg_url=True)

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


In [4]:
name_resolver.batch_lookup(['NPM1', 'NRAS','BCL2'], only_taxa='NCBITaxon:9606')


{'NPM1': TranslatorNode(curie='NCBIGene:4869', label='NPM1', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachineMixin', 'biolink:Protein', 'biolink:GeneProductMixin', 'biolink:Polypeptide', 'biolink:ChemicalEntityOrProteinOrPolypeptide'], synonyms=None, curie_synonyms=None, attributes=None, taxa=['NCBITaxon:9606']),
 'NRAS': TranslatorNode(curie='NCBIGene:4893', label='NRAS', types=['biolink:Gene', 'biolink:GeneOrGeneProduct', 'biolink:GenomicEntity', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:PhysicalEssence', 'biolink:OntologyClass', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent', 'biolink:MacromolecularMachine

In [5]:
input_node_list = ['NCBIGene:4869', 'NCBIGene:4893', 'NCBIGene:596']


In [6]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = ['Retriever',
                    'Clinical Trials KP - TRAPI 1.5.0',
                    'Drug Approvals KP - TRAPI 1.5.0',
                    'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    'Microbiome KP - TRAPI 1.5.0',
                    'MolePro',
                    'COHD TRAPI',
                    'RTX KG2 - TRAPI 1.5.0',
                    'Text Mined Cooccurrence API',
                    'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    'CATRAX BigGIM GeneExpression Performance Phase KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
for api in APInames:
    if 'Automat' in api and api not in selected_APIlist:
        selected_APIlist.append(api)
        
# selected_APIlist = []
# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)
print(selected_metaKG.shape)

All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

(16539, 5)


In [7]:
input_node1_category = ['biolink:Gene'] # Node: this has to be in a format of biolink:xxx
input_node2_category = ['biolink:Gene']

sele_predicates = list(set(TCT.select_concept(sub_list=input_node1_category,obj_list=input_node2_category,metaKG=metaKG)))
sele_APIs = TCT.select_API(sub_list=input_node1_category,obj_list=input_node2_category,metaKG=metaKG)
API_URLs = TCT.get_Translator_API_URL(sele_APIs, APInames)


In [8]:
query_json = TCT.format_query_json(input_node_list,  # a list of identifiers for input node1
                                   [],  # it can be empty list if only want to query node1
                                   input_node1_category,  # a list of categories of input node1
                                   input_node2_category,  # a list of categories of input node2
                                   sele_predicates) # a list of predicates

In [13]:
result = translator_query.parallel_api_query(query_json=query_json, select_APIs=sele_APIs, APInames = APInames, API_predicates=API_predicates, max_workers=len(API_URLs))
pairs_found = TCT.get_pair_annotation(result, input_node_list)
edge_list = TCT.parse_pair_annotation(pairs_found,input_node_list)


Microbiome KP - TRAPI 1.5.0: Success!
Automat-genome-alliance(Trapi v1.5.0): Success!
Automat-cam-kp(Trapi v1.5.0): Success!
CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0: Success!
Automat-hetionet(Trapi v1.5.0): Success!
RTX KG2 - TRAPI 1.5.0: Success!
MolePro: Success!
Automat-robokop(Trapi v1.5.0): Success!


In [ ]:
#edge_list

In [15]:
TCT.visulize_path(TCT.get_curie("NPM1"), "NCBIGene:3320", TCT.get_curie("FLT3"), result, result)

CytoscapeWidget(cytoscape_layout={'name': 'cola', 'title': 'Path', 'nodeSpacing': 80, 'edgeLengthVal': 50}, cy…

,Subject,Object,Predicates,Subject_name,Object_name
0,NCBIGene:4869,NCBIGene:3320,physically_interacts_with::infores:biothings-m...,NPM1,HSP90AA1
1,NCBIGene:3320,NCBIGene:4869,genetically_interacts_with::infores:hetionet,HSP90AA1,NPM1
2,NCBIGene:3320,NCBIGene:4869,associated_with::infores:string,HSP90AA1,NPM1
3,NCBIGene:3320,NCBIGene:4869,genetically_interacts_with::infores:automat-ro...,HSP90AA1,NPM1
4,NCBIGene:4869,NCBIGene:3320,affects::infores:automat-robokop,NPM1,HSP90AA1


In [ ]:
# results can be visualized in cytoscpe or networkx